In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\data\Telco-Customer-Churn.csv')

In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [3]:
df = df.drop("customerID", axis=1)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df_encoded = pd.get_dummies(df, drop_first=True)

In [4]:
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]


In [6]:
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]

numerical_vif = pd.DataFrame({
    "feature": numerical_features,
    "VIF": [
        variance_inflation_factor(X[numerical_features].values, i)
        for i in range(len(numerical_features))
    ]
})

numerical_vif["high_vif"] = numerical_vif["VIF"] >= 5

print("VIF for numerical features:")
print(numerical_vif.to_string(index=False))

VIF for numerical features:
       feature      VIF  high_vif
        tenure 5.836728      True
MonthlyCharges 3.216730     False
  TotalCharges 9.510931      True


In [5]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_df = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

vif_df = vif_df.sort_values("VIF", ascending=False).reset_index(drop=True)
vif_df["high_vif"] = vif_df["VIF"] >= 5

print("Variance Inflation Factor (VIF):")
print(vif_df.to_string(index=False))
print("\nFeatures with VIF >= 5:")
print(vif_df.loc[vif_df["high_vif"], ["feature", "VIF"]].to_string(index=False))

C:\Users\abhin\AppData\Local\Temp\ipykernel_15036\39930547.py:5: UserWarning: The design matrix is poorly conditioned (condition number=2.25e+19). VIF calculations may be numerically unstable.
  "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\statsmodels\stats\outliers_in

Variance Inflation Factor (VIF):
                              feature          VIF  high_vif
       MultipleLines_No phone service 1.000800e+15      True
                     PhoneService_Yes 1.000800e+15      True
                   InternetService_No 1.000800e+15      True
   OnlineSecurity_No internet service 1.000800e+15      True
 DeviceProtection_No internet service 1.000800e+15      True
     OnlineBackup_No internet service 1.000800e+15      True
      TechSupport_No internet service 1.000800e+15      True
      StreamingTV_No internet service 1.000800e+15      True
  StreamingMovies_No internet service 1.000800e+15      True
                       MonthlyCharges 8.650621e+02      True
          InternetService_Fiber optic 1.482634e+02      True
                  StreamingMovies_Yes 2.411025e+01      True
                      StreamingTV_Yes 2.405683e+01      True
                         TotalCharges 1.079373e+01      True
                               tenure 7.527280e+00  

c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


In [8]:
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42))
])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_prob = np.zeros(X_train.shape[0])

for train_idx, val_idx in cv.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    pipeline_lr.fit(X_tr, y_tr)
    oof_prob[val_idx] = pipeline_lr.predict_proba(X_val)[:, 1]
    

In [6]:
oof_prob

array([0.38202166, 0.2934312 , 0.03903892, ..., 0.55979095, 0.03344909,
       0.15583269], shape=(5634,))

In [7]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

thresholds = np.round(np.arange(0.05, 0.91, 0.05), 2)

results = []

for threshold in thresholds:

    oof_pred = (oof_prob >= threshold).astype(int)

    precision = precision_score(y_train, oof_pred)
    recall = recall_score(y_train, oof_pred)
    f1 = f1_score(y_train, oof_pred)

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    by="f1",
    ascending=False,
    inplace=True
)

results_df

c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,threshold,precision,recall,f1
5,0.30,0.541038,0.767224,0.634578
6,0.35,0.568617,0.715050,0.633481
4,0.25,0.508962,0.816722,0.627119
7,0.40,0.593900,0.664214,0.627092
8,0.45,0.623440,0.601338,0.612189
3,0.20,0.475483,0.856187,0.611416
9,0.50,0.653045,0.545151,0.594240
2,0.15,0.438664,0.913712,0.592753
10,0.55,0.687441,0.486957,0.570086
1,0.10,0.400114,0.940468,0.561389


In [8]:
best_row = results_df.loc[
    results_df["f1"].idxmax()
]

best_threshold = best_row["threshold"]

print(f"Best Threshold: {best_threshold}, Best F1 Score: {best_row['f1']:.4f}")

Best Threshold: 0.3, Best F1 Score: 0.6346


In [9]:
final_model = pipeline_lr.fit(X_train, y_train)

test_prob = final_model.predict_proba(X_test)[:, 1]

In [10]:
y_pred = (test_prob >= best_threshold).astype(int)

In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, test_prob)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")
print(f"Confusion Matrix:\n{cm}")

Accuracy: 0.7495
Precision: 0.5193
Recall: 0.7540
F1 Score: 0.6150
ROC AUC Score: 0.8418
Confusion Matrix:
[[774 261]
 [ 92 282]]


In [12]:
final_model.named_steps['classifier'].coef_

array([[ 0.05307329, -1.23652765, -0.92015294,  0.51428491,  0.01109436,
         0.01086633, -0.10355019,  0.00679378, -0.00679378,  0.21616694,
         0.7761541 , -0.09276107, -0.09276107, -0.1235607 , -0.09276107,
        -0.01202194, -0.09276107,  0.05304806, -0.09276107, -0.10050155,
        -0.09276107,  0.25714408, -0.09276107,  0.25722659, -0.2855086 ,
        -0.58685933,  0.18203427, -0.01303919,  0.18110348,  0.03226232]])

In [13]:
coefficients = pd.Series(
    final_model.named_steps["classifier"].coef_[0],
    index=X_train.columns
)

print("Top 5 positive coefficients:")
print(coefficients.nlargest(5))

print("\nTop 5 negative coefficients:")
print(coefficients.nsmallest(5))

Top 5 positive coefficients:
InternetService_Fiber optic    0.776154
TotalCharges                   0.514285
StreamingMovies_Yes            0.257227
StreamingTV_Yes                0.257144
MultipleLines_Yes              0.216167
dtype: float64

Top 5 negative coefficients:
tenure               -1.236528
MonthlyCharges       -0.920153
Contract_Two year    -0.586859
Contract_One year    -0.285509
OnlineSecurity_Yes   -0.123561
dtype: float64


In [14]:
odds_ratios = np.exp(coefficients)

In [17]:
feature_effects = pd.DataFrame({
    "coefficient": coefficients,
    "odds_ratio": odds_ratios
}).sort_values("odds_ratio", ascending=False)

print("Coefficients and odds ratios for all features, sorted by odds ratio:")
print(feature_effects.to_string())

Coefficients and odds ratios for all features, sorted by odds ratio:
                                       coefficient  odds_ratio
InternetService_Fiber optic               0.776154    2.173099
TotalCharges                              0.514285    1.672442
StreamingMovies_Yes                       0.257227    1.293338
StreamingTV_Yes                           0.257144    1.293231
MultipleLines_Yes                         0.216167    1.241310
PaperlessBilling_Yes                      0.182034    1.199655
PaymentMethod_Electronic check            0.181103    1.198539
SeniorCitizen                             0.053073    1.054507
DeviceProtection_Yes                      0.053048    1.054480
PaymentMethod_Mailed check                0.032262    1.032788
gender_Male                               0.011094    1.011156
Partner_Yes                               0.010866    1.010926
PhoneService_Yes                          0.006794    1.006817
MultipleLines_No phone service           -0.00679

In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

threshold = 0.30

# Model A: all features, using the existing fitted model and test split.
model_a = final_model
model_a_prob = model_a.predict_proba(X_test)[:, 1]
model_a_pred = (model_a_prob >= threshold).astype(int)

# Model B: remove TotalCharges while keeping the same split and preprocessing.
X_train_model_b = X_train.drop(columns=["TotalCharges"])
X_test_model_b = X_test.drop(columns=["TotalCharges"])

model_b = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(random_state=42))
])
model_b.fit(X_train_model_b, y_train)
model_b_prob = model_b.predict_proba(X_test_model_b)[:, 1]
model_b_pred = (model_b_prob >= threshold).astype(int)

def evaluate_model(model_name, y_true, y_pred, y_prob):
    return {
        "model": model_name,
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

comparison_df = pd.DataFrame([
    evaluate_model("With TotalCharges", y_test, model_a_pred, model_a_prob),
    evaluate_model("Without TotalCharges", y_test, model_b_pred, model_b_prob)
]).set_index("model")

print(f"Evaluation threshold: {threshold:.2f}")
print("\nModel performance comparison:")
print(comparison_df.round(4))

print("\nModel A confusion matrix:")
print(confusion_matrix(y_test, model_a_pred))

print("\nModel B confusion matrix:")
print(confusion_matrix(y_test, model_b_pred))

features_to_compare = [
    "tenure",
    "MonthlyCharges",
    "Contract_One year",
    "Contract_Two year",
    "InternetService_Fiber optic",
    "InternetService_No"
]

model_a_coefficients = pd.Series(
    model_a.named_steps["classifier"].coef_[0],
    index=X_train.columns
)
model_b_coefficients = pd.Series(
    model_b.named_steps["classifier"].coef_[0],
    index=X_train_model_b.columns
)

coefficient_comparison = pd.DataFrame({
    "Model A coefficient": model_a_coefficients.reindex(features_to_compare),
    "Model A odds ratio": np.exp(model_a_coefficients.reindex(features_to_compare)),
    "Model B coefficient": model_b_coefficients.reindex(features_to_compare),
    "Model B odds ratio": np.exp(model_b_coefficients.reindex(features_to_compare))
})

print("\nCoefficient and odds-ratio comparison:")
print(coefficient_comparison.round(4))

Evaluation threshold: 0.30

Model performance comparison:
                      precision  recall      f1  roc_auc
model                                                   
With TotalCharges        0.5193   0.754  0.6150   0.8418
Without TotalCharges     0.5198   0.738  0.6099   0.8388

Model A confusion matrix:
[[774 261]
 [ 92 282]]

Model B confusion matrix:
[[780 255]
 [ 98 276]]

Coefficient and odds-ratio comparison:
                             Model A coefficient  Model A odds ratio  \
tenure                                   -1.2365              0.2904   
MonthlyCharges                           -0.9202              0.3985   
Contract_One year                        -0.2855              0.7516   
Contract_Two year                        -0.5869              0.5561   
InternetService_Fiber optic               0.7762              2.1731   
InternetService_No                       -0.0928              0.9114   

                             Model B coefficient  Model B odds ratio 